In [1]:
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
original = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [3]:
train.columns

Index(['id', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [4]:
original.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [5]:
test.columns

Index(['id', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges'],
      dtype='object')

In [3]:
if 'customerID' in original:
    original.drop('customerID', axis=1, inplace=True)

if 'Churn' in original and original['Churn'].dtype == 'object':
    original['Churn'] = original['Churn'].map({'Yes': 1, 'No': 0})

if 'TotalCharges' in original and original['TotalCharges'].dtype == 'object':
    original['TotalCharges'] = pd.to_numeric(original['TotalCharges'], errors='coerce')

if 'id' in train:
    train.drop('id', axis=1, inplace=True)

if 'id' in test:
    test.drop('id', axis=1, inplace=True)

In [4]:
cats = train.select_dtypes(include='object').columns.tolist()
nums = train.select_dtypes(exclude='object').columns.tolist()
target = 'Churn'
cats.remove(target)

In [8]:
nums

['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

In [5]:
# Interaction features
if 'SeniorCitizen' in nums:
    nums.remove('SeniorCitizen')
    cats.append('SeniorCitizen')
    
new_features = []

import itertools

for i, df in enumerate([train, test]):
    for col1, col2 in itertools.combinations(nums, 2):
        mul_feature = f'{col1}_*_{col2}'
        df[mul_feature] = df[col1] * df[col2]
        diff_feature = f'{col1}_-_{col2}'
        df[diff_feature] = df[col1] - df[col2]
        div_feature = f'{col1}_/_{col2}'
        df[div_feature] = df[col1] / (df[col2] + 1)
        sum_feature = f'{col1}_+_{col2}'
        df[sum_feature] = df[col1] + df[col2]
        if i == 0:
            new_features.append(mul_feature)
            new_features.append(diff_feature)
            new_features.append(div_feature)
            new_features.append(sum_feature)

for i, df in enumerate([train, test]):
    for col in nums:
        df[f'{col}^2'] = df[col] ** 2
        if i == 0:
            new_features.append(f'{col}^2')

print('Number of new features: ', len(new_features))

Number of new features:  15


In [6]:
# Logarithmic
for i, df in enumerate([train, test]):
    for col in nums:
        log_feature = f'Log_{col}'
        df[log_feature] = np.log(df[col])

        log1p_feature = f'Log1p_{col}'
        df[log1p_feature] = np.log1p(df[col])

        sqrt_feature = f'Sqrt_{col}'
        df[sqrt_feature] = np.sqrt(df[col])

        reciprocal_feature = f'Reciprocal_{col}'
        df[reciprocal_feature] = np.reciprocal(df[col])

        if i == 0:
            new_features.extend([log_feature, log1p_feature, sqrt_feature, reciprocal_feature])

print('Number of new features: ', len(new_features))

Number of new features:  27


In [7]:
# Non-linear transforms
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

for i, df in enumerate([train, test]):
    for col in nums:
        sigmoid_feature = f'Sigmoid_{col}'
        df[sigmoid_feature] = sigmoid(df[col])

        tanh_feature = f'Tanh_{col}'
        df[tanh_feature] = np.tanh(df[col])

        # exp_feature = f'Exp_{col}'
        # df[exp_feature] = np.exp(df[col])

        if i == 0:
            new_features.extend([sigmoid_feature, tanh_feature])

print('Number of new features: ', len(new_features))

Number of new features:  33


In [8]:
# Binning !! Не границы возвращать
for col in nums:
    q1 = train[col].quantile(0.25)
    q2 = train[col].quantile(0.5)
    q3 = train[col].quantile(0.75)

    outlier_feature = f'Outlier_{col}'
    quartile_feature = f'Quartile_{col}'

    bins = [-np.inf, q1, q2, q3, np.inf]

    train[outlier_feature] = ((train[col] < q1) | (train[col] > q3)).astype(int)
    # train[quartile_feature] = pd.cut(train[col], bins=bins)

    test[outlier_feature] = ((test[col] < q1) | (test[col] > q3)).astype(int)
    # test[quartile_feature] = pd.cut(test[col], bins=bins)

    new_features.extend([outlier_feature, quartile_feature])

print('Number of new features: ', len(new_features))

Number of new features:  39


In [13]:
# for col1 in nums:

#     for col2 in cats:

#         mean_feature = f'Mean_{col1}_by_{col2}'
#         result = train.groupby(col2)[col1].mean()
#         train[mean_feature] = result
#         test[mean_feature] = result
#         new_features.append(mean_feature)

# # mean encoding
# # smoothed mean
# # leave-one-out
# # K-fold target encoding
# # Bayesian encoding
# print('Number of new features: ', len(new_features))

In [9]:
for col in cats:

    count_feature = f'Count_{col}'
    freq_feature = f'Freq_{col}'
    norm_count_feature = f'Norm_count_{col}'

    counts = train[col].value_counts()

    train[count_feature] = train[col].map(counts)
    test[count_feature] = test[col].map(counts)

    train[freq_feature] = train[count_feature] / len(train)
    test[freq_feature] = test[count_feature] / len(train)

    train[norm_count_feature] = train[count_feature] / train[count_feature].max()
    test[norm_count_feature] = test[count_feature] / train[count_feature].max()

    new_features.extend([count_feature, freq_feature, norm_count_feature])

print('Number of new features:', len(new_features))

Number of new features: 87


In [15]:
agg_funcs = [
    'mean',
    'median',
    'std',
    'min',
    'max',
    'sum',
    'count',
    'nunique',
    'skew'
]

for cat in cats:
    
    for num in nums:

        group = train.groupby(cat)[num].agg(agg_funcs)

        for func in agg_funcs:

            agg_feature = f'{num}_{func}_by_{cat}'

            train[agg_feature] = train[cat].map(group[func])
            test[agg_feature] = test[cat].map(group[func])

            new_features.append(agg_feature)

print('Number of new features:', len(new_features))

Number of new features: 519


In [16]:
for cat in cats:
    
    for num in nums:

        group_mean = train.groupby(cat)[num].mean()
        group_std = train.groupby(cat)[num].std()
        group_max = train.groupby(cat)[num].max()
        group_sum = train.groupby(cat)[num].sum()

        minus_feature = f'{num}_minus_mean_by_{cat}'

        train[minus_feature] = train[num] - train[cat].map(group_mean)
        test[minus_feature] = test[num] - test[cat].map(group_mean)

        div_feature = f'{num}_div_mean_by_{cat}'

        train[div_feature] = train[num] / (train[cat].map(group_mean) + 1e-6)
        test[div_feature] = test[num] / (test[cat].map(group_mean) + 1e-6)

        z_feature = f'{num}_zscore_by_{cat}'

        train[z_feature] = (
            train[num] - train[cat].map(group_mean)
        ) / (train[cat].map(group_std) + 1e-6)

        test[z_feature] = (
            test[num] - test[cat].map(group_mean)
        ) / (test[cat].map(group_std) + 1e-6)

        max_feature = f'{num}_div_max_by_{cat}'

        train[max_feature] = train[num] / (train[cat].map(group_max) + 1e-6)
        test[max_feature] = test[num] / (test[cat].map(group_max) + 1e-6)

        sum_feature = f'{num}_div_sum_by_{cat}'

        train[sum_feature] = train[num] / (train[cat].map(group_sum) + 1e-6)
        test[sum_feature] = test[num] / (test[cat].map(group_sum) + 1e-6)

        rank_feature = f'{num}_rank_by_{cat}'

        train[rank_feature] = train.groupby(cat)[num].rank(pct=True)
        test[rank_feature] = test.groupby(cat)[num].rank(pct=True)

        new_features.extend([minus_feature, div_feature, z_feature, max_feature, sum_feature, rank_feature])

In [10]:
# number of active services
# number of features > threshold
# is_high
# is_low
# is_missing
# condition flags
for df in [train, test]:

    df["IsMonthToMonth"] = (df["Contract"] == "Month-to-month").astype(int)

    df["IsElectronicCheck"] = (df["PaymentMethod"] == "Electronic check").astype(int)
    df["AutoPayment"] = df["PaymentMethod"].str.contains("automatic").astype(int)

    df["SeniorCitizenMonthlyInteraction"] = (
        df["SeniorCitizen"].astype(int) * df["MonthlyCharges"]
    )

    df["ServiceCount"] = (
        (df["OnlineSecurity"] == "Yes").astype(int) +
        (df["OnlineBackup"] == "Yes").astype(int) +
        (df["DeviceProtection"] == "Yes").astype(int) +
        (df["TechSupport"] == "Yes").astype(int) +
        (df["StreamingTV"] == "Yes").astype(int) +
        (df["StreamingMovies"] == "Yes").astype(int)
    )

    df["IsFiber"] = (df["InternetService"] == "Fiber optic").astype(int)

    df["FiberHighCharge"] = (
        (df["InternetService"] == "Fiber optic") &
        (df["MonthlyCharges"] > df["MonthlyCharges"].median())
    ).astype(int)

    df["FiberNoSupport"] = (
        (df["InternetService"] == "Fiber optic") &
        (df["TechSupport"] == "No")
    ).astype(int)

    df["EarlyTenure"] = (df["tenure"] < 12).astype(int)
    df["HighChargeEarly"] = (
        (df["tenure"] < 12) &
        (df["MonthlyCharges"] > df["MonthlyCharges"].median())
    ).astype(int)

    df["VeryEarly"] = (df["tenure"] < 6).astype(int)

    df["ChargeRank"] = df["MonthlyCharges"].rank(pct=True)
    df["TenureRank"] = df["tenure"].rank(pct=True)

    df["Contract_Internet"] = (
        df["Contract"].astype(str) + "_" +
        df["InternetService"].astype(str)
    )

    df["Contract_Payment"] = (
        df["Contract"].astype(str) + "_" +
        df["PaymentMethod"].astype(str)
    )

In [11]:
train.shape, test.shape

((594194, 119), (254655, 118))

In [12]:
new_cats = train.select_dtypes(include='object').columns

In [13]:
X, y = train.drop('Churn', axis=1), train['Churn']

In [14]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

label_cols = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling'
]

onehot_cols = ['PaymentMethod', 'Contract_Internet', 'Contract_Payment']

ord_mapping = {
    'Contract': ['Two year', 'One year', 'Month-to-month'],
    'MultipleLines': ['No phone service', 'No', 'Yes'],
    'InternetService': ['No', 'DSL', 'Fiber optic'],
    'OnlineSecurity': ['No internet service', 'No', 'Yes'],
    'OnlineBackup': ['No internet service', 'No', 'Yes'],
    'DeviceProtection': ['No internet service', 'No', 'Yes'],
    'TechSupport': ['No internet service', 'No', 'Yes'],
    'StreamingTV': ['No internet service', 'No', 'Yes'],
    'StreamingMovies': ['No internet service', 'No', 'Yes'],
    'Partner': ['No', 'Yes'],
    'Dependents': ['No', 'Yes'],
    'PhoneService': ['No', 'Yes'],
    'PaperlessBilling': ['No', 'Yes'],
    'gender': ['Male', 'Female']
}

ord_cols = list(ord_mapping.keys())

preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(drop='first', sparse_output=False), onehot_cols),
        ('ordinal', OrdinalEncoder(categories=[ord_mapping[col] for col in ord_cols]), ord_cols)
    ],
    remainder='passthrough'
)

pipeline = Pipeline([
    ('preprocessor', preprocessor)
])

pipeline.set_output(transform="pandas")

X_train_encoded = pipeline.fit_transform(X)
X_test_encoded = pipeline.transform(test)

In [15]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()

TOKEN = os.getenv('TELEGRAM_TOKEN')
CHAT_ID = os.getenv('TELEGRAM_CHAT_ID')

def send_telegram(msg):
    url = f'https://api.telegram.org/bot{TOKEN}/sendMessage'
    try:
        requests.post(
            url,
            data={
                'chat_id': CHAT_ID,
                'text': msg
            }
        )
    except:
        pass

In [20]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

import json

with open('lightgbm-params.json', 'r') as f:
    params = json.load(f)

seeds = [69, 228, 1337]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=420)
oof_preds = np.zeros(len(X_train_encoded))
test_preds = np.zeros(len(X_test_encoded))

for seed in seeds:

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train_encoded, y)):

        fold_test_preds = np.zeros(len(X_test_encoded))
        print(f'Fold {fold+1} | Seed {seed}')

        X_train, X_valid = X_train_encoded.iloc[train_idx], X_train_encoded.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        model = lgb.LGBMClassifier(
            n_estimators=5000,
            **params,
            verbose=-1,
            random_state=42
        )

        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            eval_metric="auc",
            callbacks=[lgb.early_stopping(200), lgb.log_evaluation(0)]
        )

        oof_preds[valid_idx] += model.predict_proba(X_valid)[:, 1] / len(seeds)
        fold_test_preds += model.predict_proba(X_test_encoded)[:, 1] / skf.n_splits

        score = roc_auc_score(y_valid, oof_preds[valid_idx])
        send_telegram(f'Fold {fold+1} | Seed {seed} | AUC: {score:.5f}')

    test_preds += fold_test_preds / len(seeds)

Fold 1 | Seed 69
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[1030]	valid_0's auc: 0.916776	valid_0's binary_logloss: 0.297068
Fold 2 | Seed 69
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[1206]	valid_0's auc: 0.917122	valid_0's binary_logloss: 0.296531
Fold 3 | Seed 69
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[1362]	valid_0's auc: 0.917611	valid_0's binary_logloss: 0.295834
Fold 4 | Seed 69
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[1518]	valid_0's auc: 0.916946	valid_0's binary_logloss: 0.296724
Fold 5 | Seed 69
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[1161]	valid_0's auc: 0.916082	valid_0's binary_logloss: 0.298127
Fold 1 | Seed 228
Training until validation scores don't improve for 200 rounds
Early stopping, best

In [21]:
send_telegram(roc_auc_score(y, oof_preds))

In [19]:
submission = pd.read_csv('submission.csv')
submission['Churn'] = fold_test_preds
submission.to_csv('submission.csv', index=False)

In [18]:
# Box-Cox
# Yeo-Johnson
# quantile transform
# standardization
# normalization

# Признаки из методов снижения размерности.

# Типы:
# 	•	Principal Component Analysis

# Использование кластеризации.

# Типы:
# 	•	K-Means clustering
# 	•	DBSCAN
# 	•	Gaussian Mixture Model

# bi-grams
# tri-grams

# entropy
# unique ratio
# diversity index

# label encoding
# ordinal encoding
# one-hot encoding
# hash encoding
# binary encoding
# WOE encoding